In [6]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import time
from tqdm import tqdm
from difflib import SequenceMatcher
from multiprocessing import Pool

import numpy as np
import pandas as pd

from transformers import pipeline, get_scheduler
from transformers import AutoTokenizer, AutoConfig, DataCollatorWithPadding
from transformers import DataCollatorForLanguageModeling, GPT2LMHeadModel,  GPT2Tokenizer, GPT2Config
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

import torch

from transformers import create_optimizer, AdamWeightDecay
from transformers import TFAutoModelForCausalLM

import datasets
from datasets import ClassLabel, load_dataset, Dataset, DatasetDict, load_metric

#print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [7]:
main_dir = './'

class CFG:
    report_to = None
    lab_assignment = 5
    wandb_kernel = "temuujin"

    debug = False
    num_workers = 12

    tokenizer_name = 'bayartsogt/mongolian-gpt2'
    model_name = 'bayartsogt/mongolian-gpt2'

    project = 'NUM-Machine-Learning-Lab-5'
    name = "Lab 5"

    config = {
        "output_dir": f"{main_dir}/lab5_finetune_gpt2",
        "group": model_name,
        "learning_rate": 2e-5,
        "weight_decay": 1e-2,
        'num_train_epochs': 10,
        "train_batch_size": 32,
        "eval_batch_size": 32,
        "dataloader_num_workers": num_workers,
        "finetuning_task": 'ner',
        "evaluation_strategy": 'epoch',
        "logging_strategy": 'epoch',
        "overwrite_output_dir": True,
        "push_to_hub": False,
    }

    model_save_dir = f"{main_dir}/lab5_gpt2_tiin_ylgal"

    test_size = 0.2

    train = True
    eval = True

    eval_metric = "seqeval"

    early_stopping_patience = 15

if CFG.debug:
    CFG.config['num_train_epochs'] = 2

config = CFG.config

In [8]:
def concatenate_columns(example):
    example["prompt"] = example["prompt"] + " " + example["answer"]
    return example

In [9]:
start_time = time.time()

In [10]:
df = pd.read_parquet(f"{main_dir}/clean_data.parquet").reset_index(drop = True)
df

,prompt,answer,length_greater,length_equal,answer_text,answer_something,answer_2
0,багтраагаа,багтраа <voc>,True,False,багтраа,<voc>,аа <voc>
1,дэнсээс,дэнс <abl>,True,False,дэнс,<abl>,ээс <abl>
2,тархалтыг,тархалт <acc>,True,False,тархалт,<acc>,ыг <acc>
3,дуулийг,дууль <acc>,True,False,дууль,<acc>,ийг <acc>
4,гуалингаар,гуалин <ins>,True,False,гуалин,<ins>,аар <ins>
...,...,...,...,...,...,...,...
14351,үтрээний,үтрээ <gen>,True,False,үтрээ,<gen>,ий <gen>
14352,лавлагаанд,лавлагаа <dat>,True,False,лавлагаа,<dat>,д <dat>
14353,эзэмшлээ,эзэмшил <voc>,True,False,эзэмшил,<voc>,ээ <voc>
14354,улстай,улс <com>,True,False,улс,<com>,тай <com>


In [11]:
df['prompt'] = '<s> bb: ' + df['prompt']
df['answer'] = df['answer'] + '</s>'

df.drop(columns = ['answer_2', 'length_greater', 'length_equal', 'answer_text', 'answer_something'], inplace = True)

infl_dataset = Dataset.from_pandas(df)
ds_train_devtest = infl_dataset.train_test_split(test_size = 0.025, seed = 42)
ds_devtest = ds_train_devtest['test'].train_test_split(test_size = 0.5, seed = 42)

ds_splits = DatasetDict({
    'train': ds_train_devtest['train'],
    'valid': ds_devtest['train'],
    'test': ds_devtest['test']
})

ds_splits["train"] = ds_splits["train"].map(concatenate_columns)
ds_splits["valid"] = ds_splits["valid"].map(concatenate_columns)

ds_splits = ds_splits.flatten()

ds_splits

Map:   0%|          | 0/13997 [00:00<?, ? examples/s]

Map:   0%|          | 0/179 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 13997
    })
    valid: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 179
    })
    test: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 180
    })
})

In [13]:
block_size = 64
tokenizer = AutoTokenizer.from_pretrained(CFG.tokenizer_name)

def preprocess_function(examples, tokenizer = tokenizer):
    return tokenizer(examples["prompt"])

tokenized_ds = ds_splits.map(
    preprocess_function,
    batched = True,
    num_proc = 4,
    remove_columns = ds_splits["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/13997 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/179 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

In [15]:
def group_texts(examples, block_size = block_size):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_ds.map(group_texts, batched = True, num_proc = 4)

Map (num_proc=4):   0%|          | 0/13997 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/179 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

#  1. Үндэс үйл үг ялгах болон ялгасан үндэс үйл үгийг ашиглан тийн ялгал ялгах

In [16]:
test_ver = 2
OUTPUT_MODEL = os.path.join(CFG.model_save_dir, f"test_v{test_ver}")

In [17]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained("bayartsogt/mongolian-gpt2")
model.resize_token_embeddings(len(tokenizer))

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
optimizer = torch.optim.Adam(model.parameters())

training_args = TrainingArguments(
    output_dir = OUTPUT_MODEL,
    evaluation_strategy = "epoch",
    learning_rate = 3e-5,
    warmup_steps = 500,
    weight_decay = 0.2,
    push_to_hub = False,
    logging_dir = "./logs",
    logging_steps = 50,
    fp16 = True,
    gradient_accumulation_steps = 1,
    max_grad_norm = 1.0,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    num_train_epochs = 10,
    logging_strategy = "steps",
    save_total_limit = 2,
    save_strategy = "no")

scheduler = get_scheduler(
    name = "cosine",
    optimizer = optimizer,
    num_warmup_steps = training_args.warmup_steps,
    num_training_steps = training_args.num_train_epochs * (len(lm_dataset["train"]) // training_args.per_device_train_batch_size)
)

In [18]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = lm_dataset["train"],
    eval_dataset = lm_dataset["valid"],
    data_collator = data_collator,
    optimizers = (optimizer, scheduler)
)

trainer.train()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: temuujin-razy. Use `wandb login --relogin` to force relogin


  0%|          | 0/3220 [00:00<?, ?it/s]

In [ ]:
generator = pipeline("text-generation", model = model, tokenizer = tokenizer, num_beams = 5)

prompt = "<s> bb: гүйцэтгэлийг"
ans = generator(prompt, max_length = 15)
print(ans[0]['generated_text'].split('bb:')[1].strip())

gc.collect()

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


гүйцэтгэлийг гүйцэтгэл <acc>


30

In [ ]:
test_dataset = ds_splits["test"]

batch_size = 10
all_prompts = [sample['prompt'] for sample in test_dataset]
all_expected_answers = [sample['answer'] for sample in test_dataset]

total = 0
correct = 0
correct2 = 0
num_batches = len(all_prompts) // batch_size + (1 if len(all_prompts) % batch_size != 0 else 0)

for i in tqdm(range(0, len(all_prompts), batch_size), total=num_batches, desc="Evaluating"):
    batch_prompts = all_prompts[i:i + batch_size]
    expected_answers = all_expected_answers[i:i + batch_size]
    results = generator(batch_prompts, max_length=15)
    for j, result_list in enumerate(results):
        result = result_list[0]
        total += 1
        generated_text = result['generated_text']
        expected_answer = expected_answers[j].replace("</s>", "").strip()
        w = expected_answer.split()
        words = generated_text.split()
        if w[0] == words[3]:
            correct += 1

        if w[1] == words[4]:
            correct2 += 1
accuracy = correct / total * 100
accuracy2 = correct2 / total * 100

print(f"\nAccuracy: {accuracy:.2f}%")
print(f"Accuracy: {accuracy2:.2f}%")

Evaluating: 100%|██████████| 18/18 [03:16<00:00, 10.92s/it]


Accuracy: 81.11%
Accuracy: 98.33%


In [ ]:
torch.cuda.empty_cache()
gc.collect()

180

In [ ]:
del model, tokenizer, trainer, tokenized_ds, lm_dataset
gc.collect()

0

# 2. Тийн ялгал ялгах модел тусад нь сургах

In [ ]:
df = pd.read_parquet(f"{main_dir}/clean_data.parquet").reset_index(drop = True)

df['answer'] = df['answer_2']
df.drop(columns = ['answer_2', 'length_greater', 'length_equal', 'answer_text', 'answer_something'], inplace = True)

df['prompt'] = '<s> bb: ' + df['prompt']
df['answer'] = df['answer'] + '</s>'

infl_dataset = Dataset.from_pandas(df)
ds_train_devtest = infl_dataset.train_test_split(test_size = 0.025, seed = 42)
ds_devtest = ds_train_devtest['test'].train_test_split(test_size = 0.5, seed = 42)

ds_splits = DatasetDict({
    'train': ds_train_devtest['train'],
    'valid': ds_devtest['train'],
    'test': ds_devtest['test']
})

ds_splits["train"] = ds_splits["train"].map(concatenate_columns)
ds_splits["valid"] = ds_splits["valid"].map(concatenate_columns)

ds_splits = ds_splits.flatten()

Map:   0%|          | 0/13997 [00:00<?, ? examples/s]

Map:   0%|          | 0/179 [00:00<?, ? examples/s]

In [ ]:
block_size = 64
tokenizer = AutoTokenizer.from_pretrained(CFG.tokenizer_name)

def preprocess_function(examples, tokenizer = tokenizer):
    return tokenizer(examples["prompt"])

tokenized_ds = ds_splits.map(
    preprocess_function,
    batched = True,
    num_proc = 4,
    remove_columns = ds_splits["train"].column_names,
)

lm_dataset = tokenized_ds.map(group_texts, batched = True, num_proc = 4)

Map (num_proc=4):   0%|          | 0/13997 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/179 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/13997 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/179 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm = False)
model = AutoModelForCausalLM.from_pretrained(CFG.model_name)

test_ver = 2
OUTPUT_MODEL = os.path.join("tiin_ylgal_ylgah", f"test_v{test_ver}")

training_args = TrainingArguments(
    report_to = None,
    output_dir = OUTPUT_MODEL,
    num_train_epochs = 10,
    overwrite_output_dir = config["overwrite_output_dir"],
    learning_rate = config['learning_rate'],
    weight_decay = config["weight_decay"],
    evaluation_strategy = config["evaluation_strategy"],
    push_to_hub = False,
    do_eval = True,
    disable_tqdm = False,
    save_total_limit = 2,
    save_strategy = "no")

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = lm_dataset["train"],
    eval_dataset = lm_dataset["valid"],
    tokenizer = tokenizer,
    data_collator = data_collator,
    compute_metrics = None,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,1.257573
2,2.206500,1.133472
3,2.206500,1.113347
4,1.032000,1.116728
5,0.944000,1.127261
6,0.944000,1.148927
7,0.892700,1.166274
8,0.857800,1.189089
9,0.857800,1.206893
10,0.832400,1.222962


TrainOutput(global_step=3170, training_loss=1.1114187583562327, metrics={'train_runtime': 382.4064, 'train_samples_per_second': 66.134, 'train_steps_per_second': 8.29, 'total_flos': 826009436160000.0, 'train_loss': 1.1114187583562327, 'epoch': 10.0})

In [ ]:
generator = pipeline("text-generation", model = model, tokenizer = tokenizer, num_beams = 5)

prompt = "<s> bb: гүйцэтгэлийг"
ans = generator(prompt, max_length = 15)
print(ans[0]['generated_text'].split('bb:')[1].strip())

gc.collect()

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


гүйцэтгэлийг ийг <acc>


2903

In [ ]:
test_dataset = ds_splits["test"]

num_workers = 8
references = []
preds = []

def generate_prediction(args):
    prompt, answer = args
    prompt_pred = generator(prompt, max_length = 20)
    prompt_pred = prompt_pred[0]['generated_text'].split('bb:')[1].strip()
    ref_prompt = answer.split('</s>')[0].strip()
    return prompt_pred, ref_prompt

prompt_answer_pairs = zip(test_dataset['prompt'], test_dataset['answer'])
with Pool(num_workers) as pool:
    results = list(tqdm(pool.imap(generate_prediction, prompt_answer_pairs), total = len(test_dataset)))

preds, references = zip(*results)

print(preds[:10])
print(references[:10])

100%|██████████| 180/180 [04:54<00:00,  1.64s/it]


('буртагтай тай <com>', 'завсарлагын ы <gen>', 'мөнгөтэй тэй <com>', 'тэврэлтэд д <dat>', 'далайцын ын <gen>', 'баатрыг ийг <acc>', 'ганхалттай тай <com>', 'ховилыг ыг <acc>', 'залгавраас аас <abl>', 'заавраа аа <voc>')
('тай <com>', 'ын <gen>', 'тэй <com>', 'д <dat>', 'ын <gen>', 'ыг <acc>', 'тай <com>', 'ыг <acc>', 'аас <abl>', 'аа <voc>')


In [ ]:
# Exact match (nice)
print("Үг үсэггүй яг таарсан accuracy:")
(np.array([pred.split(' ', 1)[1].strip() for pred in preds]) == np.array(references)).sum()/len(preds)

Үг үсэггүй яг таарсан accuracy:


0.7444444444444445

In [ ]:
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Notebook total time: {elapsed_time/60:.5f} minutes")

Notebook total time: 21.33368 minutes
